# Vocabulary Dynamics

Analyzes the structure of the model's vocabulary representation space: embedding-unembedding alignment, subspace geometry, frequency bias, isotropy, and token neighborhoods.

**References:**
- Gao et al. (2019) "Representation Degeneration in Token Embeddings"
- Ethayarajh (2019) "How Contextual are Contextualized Word Representations?"

## Why This Matters

Vocabulary dynamics analyzes the structure of the embedding and unembedding spaces — alignment between W_E and W_U, frequency bias, isotropy, and neighborhood structure. These properties constrain what the model can represent and predict at the token level.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from irtk import HookedTransformer, HookedTransformerConfig
from irtk.vocabulary_dynamics import (
    embedding_unembed_alignment,
    vocab_subspace_analysis,
    token_frequency_bias,
    embedding_isotropy,
    token_neighborhood_structure,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
print(f"Model: d_vocab={cfg.d_vocab}, d_model={cfg.d_model}")

In [ ]:
# 1. Embedding-unembedding alignment
align = embedding_unembed_alignment(model, top_k=5)
print(f"Mean alignment: {align['mean_alignment']:.4f} ± {align['alignment_std']:.4f}")
print(f"Top aligned tokens: {align['top_aligned_tokens']}")
print(f"Bottom aligned tokens: {align['bottom_aligned_tokens']}")

In [ ]:
# 2. Vocabulary subspace analysis
subspace = vocab_subspace_analysis(model, n_components=5)
print(f"Effective rank: {subspace['effective_rank']:.2f}")
print(f"Mean embedding norm: {subspace['mean_embedding_norm']:.4f}")
print(f"\nPrincipal components:")
for i, (sv, var) in enumerate(zip(subspace['singular_values'], subspace['explained_variance_ratio'])):
    print(f"  PC{i+1}: σ={sv:.4f}, variance={var:.4f}, cumulative={subspace['cumulative_variance'][i]:.4f}")

In [ ]:
# 3. Token frequency bias
bias = token_frequency_bias(model, top_k=5)
print(f"Mean unembed norm: {bias['mean_norm']:.4f} ± {bias['norm_std']:.4f}")
print(f"Norm ratio (max/min): {bias['norm_ratio']:.2f}")
print(f"Highest norm tokens: {bias['highest_norm_tokens']}")
print(f"Lowest norm tokens: {bias['lowest_norm_tokens']}")

In [ ]:
# 4. Embedding isotropy
iso = embedding_isotropy(model)
print(f"Isotropy score: {iso['isotropy_score']:.4f} (1=perfectly isotropic)")
print(f"Mean cosine: {iso['mean_cosine']:.4f} ± {iso['std_cosine']:.4f}")
print(f"Range: [{iso['min_cosine']:.4f}, {iso['max_cosine']:.4f}]")

In [ ]:
# 5. Token neighborhood structure
query = [0, 10, 50, 99]
neighbors = token_neighborhood_structure(model, query_tokens=query, k=3)
for t in query:
    nn = neighbors['neighbors'][t]
    sims = neighbors['neighbor_similarities'][t]
    rank = neighbors['self_similarity_rank'][t]
    print(f"Token {t}: neighbors={nn}, sims=[{', '.join(f'{s:.3f}' for s in sims)}], self_rank={rank}")
print(f"\nMean neighbor similarity: {neighbors['mean_neighbor_similarity']:.4f}")